# 04.5 Refining the Temporal Feature Set

For each of the same top 25 variables, this controlled experiment removes `first` and adds `last_minus_recent3_mean`. The customer population, total feature count, split, LightGBM parameters, and evaluation remain unchanged.


In [1]:
from pathlib import Path

def find_project_root() -> Path:
    """Use the parent of the notebook working directory as the project root."""
    project_root = Path.cwd().resolve().parent
    if not (project_root / 'data').is_dir():
        raise FileNotFoundError(
            "Expected the project data directory one level above the current working directory."
        )
    return project_root

import json, platform, time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.feather as feather
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

pd.set_option('display.max_rows', 100)
print('Python:', platform.python_version())
print('LightGBM:', lgb.__version__)


Python: 3.10.19
LightGBM: 4.7.0


### 1. Define the feature replacement

In [5]:
ROOT = find_project_root()
STEP_DIR = ROOT / '04.5_temporal_refined'
SOURCE_DATA = ROOT / '04_temporal_features' / 'train_aggregated_temporal.feather'
ORIGINAL_IMPORTANCE = ROOT / '04_temporal_features' / 'original_variable_importance.csv'
OUTPUT_DATA = STEP_DIR / 'train_aggregated_temporal_refined.feather'
MODEL_PATH = STEP_DIR / 'lightgbm_temporal_refined.txt'
IMPORTANCE_PATH = STEP_DIR / 'temporal_refined_feature_importance.csv'
PREDICTION_PATH = STEP_DIR / 'temporal_refined_validation_predictions.parquet'
METRICS_PATH = STEP_DIR / 'temporal_refined_metrics.json'

for path in [SOURCE_DATA, ORIGINAL_IMPORTANCE]:
    assert path.exists(), f'Cannot find input file: {path}'

top25_table = pd.read_csv(ORIGINAL_IMPORTANCE).sort_values('original_variable_rank')
TOP25 = top25_table.head(25)['original_variable'].tolist()
FIRST_FEATURES = [f'{v}_first' for v in TOP25]
NEW_FEATURES = [f'{v}_last_minus_recent3_mean' for v in TOP25]
print('Top 25:', TOP25)
print('Deleted features:', len(FIRST_FEATURES))
print('New features:', len(NEW_FEATURES))


Top 25: ['P_2', 'B_1', 'R_1', 'D_39', 'B_2', 'B_9', 'S_3', 'D_42', 'D_44', 'B_4', 'B_3', 'B_11', 'D_41', 'D_43', 'D_46', 'R_2', 'R_3', 'B_37', 'B_7', 'D_48', 'D_49', 'D_51', 'P_3', 'D_50', 'S_23']
Deleted features: 25
New features: 25


### 2. Build and save the refined dataset

The refined table is saved as its own Feather artifact.


In [ ]:
started = time.perf_counter()
data = pd.read_feather(SOURCE_DATA)
assert data['customer_ID'].is_unique
assert all(c in data.columns for c in FIRST_FEATURES)

for variable in TOP25:
    data[f'{variable}_last_minus_recent3_mean'] = (
        data[f'{variable}_last'].astype('float32')
        - data[f'{variable}_recent_3_mean'].astype('float32')
    ).astype('float16')

data = data.drop(columns=FIRST_FEATURES)
assert not set(FIRST_FEATURES).intersection(data.columns)
assert set(NEW_FEATURES).issubset(data.columns)
assert data.shape[1] == 1024

feather.write_feather(
    pa.Table.from_pandas(data, preserve_index=False), OUTPUT_DATA,
    compression='zstd', compression_level=6, chunksize=65_536, version=2,
)
feature_generation_seconds = time.perf_counter() - started
print(f'Data: {data.shape[0]:,} customers × {data.shape[1]:,} columns')
print('Model features:', data.shape[1] - 2)
print(f'Saved: {OUTPUT_DATA} ({OUTPUT_DATA.stat().st_size / 2**30:.2f} GiB)')
print(f'Generation + save: {feature_generation_seconds:.1f} sec')


### 3. Validate the replacement

Verifies customer count and uniqueness, confirms that all 25 old columns are absent and all 25 replacements are present, and checks the arithmetic relationship for every row.


In [8]:
source_ids = pd.read_feather(SOURCE_DATA, columns=['customer_ID'])['customer_ID']
assert len(source_ids) == len(data)
assert source_ids.is_unique and data['customer_ID'].is_unique
assert np.array_equal(source_ids.to_numpy(), data['customer_ID'].to_numpy())

for variable in TOP25:
    expected = data[f'{variable}_last'].astype('float32') - data[f'{variable}_recent_3_mean'].astype('float32')
    actual = data[f'{variable}_last_minus_recent3_mean'].astype('float32')
    assert np.isclose(actual, expected, rtol=2e-3, atol=2e-3, equal_nan=True).all(), variable

print('Check passed: customer count/order/uniqueness consistent, 25 new features arithmetic relationships correct, NaNs preserved.')


Check passed: customer count/order/uniqueness consistent, 25 new features arithmetic relationships correct, NaNs preserved.


### 4. Train under the identical experimental protocol

The exact 70/30 stratified split and LightGBM configuration from step 04 are reused. This controls for sampling and model variation, so metric differences can be attributed to the feature substitution.

In [9]:
def amex_metric(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.int8)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    weights = np.where(y_true == 0, 20.0, 1.0)
    order = np.argsort(-y_pred, kind='mergesort')
    yy, ww = y_true[order], weights[order]
    top_mask = np.cumsum(ww) <= 0.04 * ww.sum()
    top_four = (yy[top_mask] * ww[top_mask]).sum() / (y_true * weights).sum()
    def weighted_gini(prediction):
        idx = np.argsort(-prediction, kind='mergesort')
        sy, sw = y_true[idx], weights[idx]
        lorentz = np.cumsum(sy * sw) / np.sum(sy * sw)
        random = np.cumsum(sw) / np.sum(sw)
        return np.sum((lorentz - random) * sw)
    return 0.5 * (weighted_gini(y_pred) / weighted_gini(y_true) + top_four)

customer_ids = data['customer_ID'].copy()
y = data['target'].astype('int8').copy()
X = data.drop(columns=['customer_ID', 'target']).copy()
X['S_2_last'] = (pd.to_datetime(X['S_2_last']) - pd.Timestamp('2018-03-01')).dt.days.astype('int16')
categorical = [f'{c}_last' for c in ['B_30','B_38','D_114','D_116','D_117','D_120','D_126','D_63','D_64','D_66','D_68']]
for column in categorical:
    X[column] = X[column].astype('category')

indices = np.arange(len(X))
train_idx, valid_idx = train_test_split(indices, train_size=.70, test_size=.30, random_state=42, stratify=y)
y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
assert X.shape[1] == 1022

model = lgb.LGBMClassifier(
    objective='binary', n_estimators=1500, learning_rate=.03,
    num_leaves=31, max_depth=-1, min_child_samples=100,
    subsample=.8, colsample_bytree=.8, reg_lambda=1.0,
    max_bin=127, random_state=42, n_jobs=4, verbosity=-1,
)
training_started = time.perf_counter()
model.fit(
    X.iloc[train_idx], y_train,
    eval_set=[(X.iloc[valid_idx], y_valid)], eval_metric='auc',
    categorical_feature=categorical,
    callbacks=[lgb.early_stopping(100, verbose=True), lgb.log_evaluation(50)],
)
training_seconds = time.perf_counter() - training_started
probability = model.predict_proba(X.iloc[valid_idx], num_iteration=model.best_iteration_)[:, 1]
result = {
    'roc_auc': float(roc_auc_score(y_valid, probability)),
    'amex_metric': float(amex_metric(y_valid, probability)),
    'best_iteration': int(model.best_iteration_),
    'training_seconds': float(training_seconds),
    'feature_count': int(X.shape[1]),
}
result


/opt/anaconda3/envs/cs178/lib/python3.10/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[50]	valid_0's auc: 0.951521	valid_0's binary_logloss: 0.288532
[100]	valid_0's auc: 0.955572	valid_0's binary_logloss: 0.243151
[150]	valid_0's auc: 0.957948	valid_0's binary_logloss: 0.230335
[200]	valid_0's auc: 0.95945	valid_0's binary_logloss: 0.224868
[250]	valid_0's auc: 0.960345	valid_0's binary_logloss: 0.22189
[300]	valid_0's auc: 0.960906	valid_0's binary_logloss: 0.2201
[350]	valid_0's auc: 0.961291	valid_0's binary_logloss: 0.218897
[400]	valid_0's auc: 0.961538	valid_0's binary_logloss: 0.218137
[450]	valid_0's auc: 0.961709	valid_0's binary_logloss: 0.21763
[500]	valid_0's auc: 0.961814	valid_0's binary_logloss: 0.217311
[550]	valid_0's auc: 0.961888	valid_0's binary_logloss: 0.217095
[600]	valid_0's auc: 0.961927	valid_0's binary_logloss: 0.21697
[650]	valid_0's auc: 0.961972	valid_0's binary_logloss: 0.216831
[700]	valid_0's auc: 0.961996	valid_0's binary_logloss: 0.216756
[750]	valid_0's auc: 0.962021	valid

{'roc_auc': 0.9622113842161928,
 'amex_metric': 0.7926643597558998,
 'best_iteration': 1451,
 'training_seconds': 645.1120789579581,
 'feature_count': 1022}

### 5. Compare performance and importance

Both ROC-AUC and the AMEX metric improve slightly. The gain is small, but its direction is consistent, so this 1,022-feature table becomes the reference dataset for model comparison, cross-validation, and tuning.


In [10]:
importance = pd.DataFrame({
    'feature': model.booster_.feature_name(),
    'gain': model.booster_.feature_importance(importance_type='gain'),
    'split': model.booster_.feature_importance(importance_type='split'),
}).sort_values(['gain','feature'], ascending=[False,True]).reset_index(drop=True)
importance['gain_share'] = importance['gain'] / importance['gain'].sum()
importance['rank'] = np.arange(1, len(importance) + 1)
importance['is_new_last_minus_recent3'] = importance['feature'].isin(NEW_FEATURES)
importance.to_csv(IMPORTANCE_PATH, index=False)

previous_04 = {
    'roc_auc': 0.962146684320197,
    'amex_metric': 0.792613685300115,
    'best_iteration': 1327,
    'training_seconds': 500.9297503330008,
    'feature_count': 1022,
}
comparison = pd.DataFrame([previous_04, result], index=['04 temporal', '04.5 refined'])
comparison.loc['04.5 - 04'] = comparison.loc['04.5 refined'] - comparison.loc['04 temporal']
display(comparison)
print('Newly added last_minus_recent3_mean features rankings:')
display(importance[importance['is_new_last_minus_recent3']].head(25))


,roc_auc,amex_metric,best_iteration,training_seconds,feature_count
04 temporal,0.962147,0.792614,1327.0,500.929750,1022.0
04.5 refined,0.962211,0.792664,1451.0,645.112079,1022.0
04.5 - 04,0.000065,0.000051,124.0,144.182329,0.0


Newly added last_minus_recent3_mean features rankings:


,feature,gain,split,gain_share,rank,is_new_last_minus_recent3
52,D_48_last_minus_recent3_mean,6528.828965,170,0.001483,53,True
60,B_9_last_minus_recent3_mean,5063.736559,123,0.001150,61,True
79,B_37_last_minus_recent3_mean,3532.256027,78,0.000802,80,True
93,B_4_last_minus_recent3_mean,2955.441047,89,0.000671,94,True
103,D_46_last_minus_recent3_mean,2664.181850,125,0.000605,104,True
169,P_2_last_minus_recent3_mean,1580.126905,84,0.000359,170,True
184,B_1_last_minus_recent3_mean,1403.167117,55,0.000319,185,True
192,B_11_last_minus_recent3_mean,1366.481745,49,0.000310,193,True
238,B_2_last_minus_recent3_mean,1072.499964,77,0.000244,239,True
245,D_39_last_minus_recent3_mean,1047.946736,65,0.000238,246,True


### 6. Save the artifacts

In [ ]:
model.booster_.save_model(MODEL_PATH, num_iteration=model.best_iteration_)
pd.DataFrame({
    'customer_ID': customer_ids.iloc[valid_idx].to_numpy(),
    'target': y_valid.to_numpy(),
    'temporal_refined_probability': probability.astype('float32'),
}).to_parquet(PREDICTION_PATH, index=False, compression='zstd')

metrics = {
    'previous_04': previous_04,
    'temporal_refined': result,
    'delta_vs_04': {k: result[k] - previous_04[k] for k in result},
    'top25_original_variables': TOP25,
    'removed_features': FIRST_FEATURES,
    'added_features': NEW_FEATURES,
    'feature_generation_seconds': feature_generation_seconds,
}
METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
for path in [OUTPUT_DATA, MODEL_PATH, IMPORTANCE_PATH, PREDICTION_PATH, METRICS_PATH]:
    print(path.name, '->', path.resolve())
